### Step 1 - Load the Silver table

In [0]:
df_gold = spark.table(
    "workspace.data_analysis.silver_employees"
)

### Step 2 - Check the data

In [0]:
display(df_gold)

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department,Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp
EMP3000,Ganesh,Reddy,ganesh.reddy@example.com,9542351161,Support,Karnataka,1967,59,2020-08-31,100252.34,Poor,Unknown,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3001,Lakshmi,Nathan,lakshmi.nathan@example.com,8255341928,HR,Punjab,2004,22,2017-02-26,113071.48,Average,Unknown,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3002,Meena,Joshi,meena.joshi@example.com,7884969653,Sales,Delhi,1985,41,2025-03-29,142863.71,Poor,True,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3003,Rahul,Bose,rahul.bose@example.com,6462704828,Operations,Rajasthan,1989,37,2017-10-26,146384.8,Excellent,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3004,Shalini,Verma,shalini.verma@example.com,7278248963,Support,Tamil Nadu,1996,30,2024-11-08,148165.17,Good,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3006,Sanjay,Reddy,sanjay.reddy@example.com,6801326773,HR,Tamil Nadu,1986,40,2017-02-23,48776.11,Average,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3007,Prakash,Sharma,prakash.sharma@example.com,7191361939,Operations,Punjab,1996,30,2025-11-11,32310.41,Average,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3010,Pooja,Patel,pooja.patel@example.com,7322602563,Sales,Kerala,1974,52,2019-02-02,137372.69,Average,Unknown,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3011,Hema,Reddy,hema.reddy@example.com,8294019655,Marketing,Telangana,1979,47,2019-06-21,66032.74,Average,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3012,Arjun,Reddy,arjun.reddy@example.com,7366299468,Support,Delhi,1969,57,2021-12-27,64239.46,Average,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z


In [0]:
print("Silver rows loaded:",df_gold.count())

Silver rows loaded: 823


In [0]:
df_gold.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone_Number: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Birth_Year: long (nullable = true)
 |-- Age: long (nullable = true)
 |-- Join_Date: date (nullable = true)
 |-- Salary: string (nullable = true)
 |-- Performance_Score: string (nullable = true)
 |-- Remote_Work: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



### Q1 - How many employees are in each department?

In [0]:
from pyspark.sql.functions import col
display(
    df_gold
    .groupBy("Department")
    .count()
    .orderBy(col("count").desc())
)

Department,count
Support,104
Marketing,89
HR,87
Finance,87
Operations,86
Legal,83
DevOps,79
IT,74
Admin,69
Sales,65


In [0]:
gold_dept_headcount = df_gold.groupBy("Department").count().orderBy(col("count").desc())

gold_dept_headcount.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.gold_dept_headcount")

In [0]:
display(
    spark.table("workspace.data_analysis.gold_dept_headcount")
)

Department,count
Support,104
Marketing,89
HR,87
Finance,87
Operations,86
Legal,83
DevOps,79
IT,74
Admin,69
Sales,65


### Q2 - What is the average salary in each department, highest first?

In [0]:
from pyspark.sql.functions import col,avg

gold_dept_avg_salary = df_gold.groupBy("Department").agg(
    avg(col("Salary").cast("double")).alias("average_salary")
).orderBy(col("average_salary").desc())


display(gold_dept_avg_salary)

Department,average_salary
IT,97040.92040540538
Admin,93737.2047826087
Support,92059.80605769233
DevOps,86336.95797468352
Sales,85371.47953846154
Marketing,85139.08910112361
HR,84326.90137931035
Legal,84248.39650602412
Operations,83758.15511627904
Finance,82831.58103448276


In [0]:
gold_dept_avg_salary.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.gold_dept_avg_salary")

In [0]:
display(
    spark.table("workspace.data_analysis.gold_dept_avg_salary")
)

Department,average_salary
IT,97040.92040540538
Admin,93737.2047826087
Support,92059.80605769233
DevOps,86336.95797468352
Sales,85371.47953846154
Marketing,85139.08910112361
HR,84326.90137931035
Legal,84248.39650602412
Operations,83758.15511627904
Finance,82831.58103448276


### Q3 - How many employees work in each region?

### Step 1 - Count employees by region

In [0]:
gold_region_headcount = df_gold.groupBy("Region").count().orderBy(col("count").desc())

display(gold_region_headcount)

Region,count
Gujarat,88
Maharashtra,87
Karnataka,86
Telangana,86
Delhi,84
Rajasthan,82
Kerala,80
Punjab,77
West Bengal,77
Tamil Nadu,76


In [0]:
gold_region_headcount.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.gold_region_headcount")

In [0]:
display(
    spark.table("workspace.data_analysis.gold_region_headcount")
)

Region,count
Gujarat,88
Maharashtra,87
Telangana,86
Karnataka,86
Delhi,84
Rajasthan,82
Kerala,80
Punjab,77
West Bengal,77
Tamil Nadu,76


### Q4 - How many employees are in each status?

In [0]:
gold_status_headcount = df_gold.groupBy("Status").count().orderBy(col("count").desc())

display(gold_status_headcount)

Status,count
Active,220
Inactive,210
On Leave,203
Pending,190


In [0]:
gold_status_headcount.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.gold_status_headcount")

### Q5 - How many "Excellent" performers does each department have?

### Step 1 - Filter Excellent performers

In [0]:
excellent_employees = df_gold.filter(
    col("Performance_Score") == "Excellent"
)

display(excellent_employees)

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department,Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp
EMP3003,Rahul,Bose,rahul.bose@example.com,6462704828,Operations,Rajasthan,1989,37,2017-10-26,146384.8,Excellent,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3015,Rajesh,Verma,rajesh.verma@example.com,8668893734,Operations,Maharashtra,1969,57,2020-03-23,76971.78,Excellent,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3016,Arun,Das,arun.das@example.com,8641708053,Finance,Rajasthan,1990,36,2015-12-10,105343.71,Excellent,True,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3017,Kiran,Sharma,kiran.sharma@example.com,8966319314,Finance,Gujarat,2001,25,2017-11-15,120862.26,Excellent,False,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3025,Pooja,Nair,pooja.nair@example.com,9773782639,Legal,Delhi,1984,42,2016-01-17,53087.58,Excellent,False,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3029,Deepa,Patel,deepa.patel@example.com,8395339421,HR,Delhi,1987,39,2018-04-24,27742.45,Excellent,Unknown,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3033,Hema,Subramaniam,hema.subramaniam@example.com,8024026811,IT,Kerala,2003,23,2016-08-15,148440.91,Excellent,Unknown,Inactive,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3035,Divya,Chatterjee,divya.chatterjee@example.com,7809885165,DevOps,West Bengal,1971,55,2017-11-10,71787.63,Excellent,True,On Leave,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3037,Karthik,Nair,karthik.nair@example.com,9438156149,DevOps,Kerala,1984,42,2021-05-13,99527.97,Excellent,False,Pending,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3042,Pooja,Das,pooja.das@example.com,9850542357,Marketing,Karnataka,1974,52,2022-05-11,112310.0,Excellent,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z


In [0]:
gold_excellent_performers = excellent_employees.groupBy("Department").count().orderBy(col("count").desc())

display(gold_excellent_performers)

Department,count
Marketing,25
IT,25
Finance,23
HR,23
Operations,23
DevOps,20
Admin,20
Support,19
Sales,17
Legal,16


In [0]:
gold_excellent_performers.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.gold_excellent_performers")

In [0]:
display(spark.table("workspace.data_analysis.gold_excellent_performers"))

Department,count
IT,25
Marketing,25
Finance,23
HR,23
Operations,23
DevOps,20
Admin,20
Support,19
Sales,17
Legal,16


### Q6 - How many people joined the company each year?

In [0]:
from pyspark.sql.functions import year

gold_yearly_joiners = (
    df_gold.withColumn(
    "Join_Year",year(col("Join_Date")))
    .groupBy("Join_Year")
    .count()
    .orderBy("Join_Year")
)

display(gold_yearly_joiners)


Join_Year,count
2015,65
2016,80
2017,66
2018,72
2019,75
2020,72
2021,68
2022,84
2023,70
2024,88


In [0]:
gold_yearly_joiners.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.data_analysis.gold_yearly_joiners"
)

### Challenges

### C1 - Top 5 Highest-Paid Employees

In [0]:
from pyspark.sql.functions import concat,lit

gold_top5_employees= df_gold.withColumn(
    "Full_Name",
    concat(col("First_Name"),lit(" "),col("Last_Name"))
).select(
    "Full_Name",
    "Department",
    "Salary"
).orderBy(
    col("Salary").cast("double").desc()
).limit(5)

display(gold_top5_employees)

Full_Name,Department,Salary
Hema Verma,Support,149998.61
Uma Das,Operations,149929.86
Kavya Chatterjee,HR,149927.56
Arjun Mehta,Sales,149873.38
Prakash Nair,Support,149862.79


In [0]:
gold_top5_employees.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.challenge_gold_top5_employees")

### C2 - Average salary of Active employees in each region

In [0]:
active_employees = df_gold.filter(
    col("Status") == "Active"
)

average_salary_region = (
    active_employees
    .groupBy("Region")
    .agg(avg(col("Salary").cast("double")).alias("Average_Salary"))
    .orderBy(col("Average_Salary").desc())
)

In [0]:
display(average_salary_region)

Region,Average_Salary
Karnataka,96821.70333333334
West Bengal,94891.29523809522
Telangana,90270.88250000002
Kerala,89801.30115384616
Gujarat,87653.62200000002
Punjab,85577.05299999999
Tamil Nadu,82203.10800000001
Rajasthan,81470.18380952382
Maharashtra,79360.733125
Delhi,78999.78208333334


In [0]:
average_salary_region.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.challenge_gold_average_salary_region")

### C3 - Poor Performers Who Are Still Active
Business question:

Which employees have a Poor performance score but are still Active, and how many are there in each department?

In [0]:
poor_performance_active_employee = df_gold.filter(
    (col("Performance_Score") == "Poor") &
    (col("Status")=="Active")
)
display(poor_performance_active_employee)

department_wise_count = poor_performance_active_employee.groupBy("Department").count().orderBy("count",ascending=False)

display(department_wise_count)

Employee_ID,First_Name,Last_Name,Email,Phone_Number,Department,Region,Birth_Year,Age,Join_Date,Salary,Performance_Score,Remote_Work,Status,batch_id,source_system,ingestion_timestamp
EMP3023,Naveen,Nair,naveen.nair@example.com,8902787429,Admin,Gujarat,1984,42,2020-08-11,92497.7,Poor,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3031,Arun,Gupta,arun.gupta@example.com,6471138267,Marketing,Karnataka,1995,31,2021-04-18,66000.79,Poor,False,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3100,Naveen,Gupta,naveen.gupta@example.com,7172233206,Support,West Bengal,2002,24,2019-05-05,127300.65,Poor,False,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3110,Ganesh,Kumar,ganesh.kumar@example.com,9393483564,Finance,Delhi,1965,61,2022-06-06,90038.65,Poor,Unknown,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3118,Meena,Verma,meena.verma@example.com,9665800439,Support,Tamil Nadu,1993,33,2023-04-17,95235.62,Poor,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3129,Anjali,Verma,anjali.verma@example.com,6388629127,IT,West Bengal,2002,24,2022-05-25,88850.61,Poor,False,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3134,Pooja,Iyer,pooja.iyer@example.com,9335658804,Support,Rajasthan,1993,33,2023-08-20,55409.16,Poor,False,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3142,Meena,Das,meena.das@example.com,7080214427,Legal,Telangana,1999,27,2024-07-27,33557.99,Poor,False,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3143,Anitha,Iyer,anitha.iyer@example.com,6367303718,Marketing,Gujarat,1967,59,2025-09-02,43260.98,Poor,True,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z
EMP3158,Priya,Nathan,priya.nathan@example.com,7476447264,Legal,Punjab,1967,59,2024-04-10,134720.87,Poor,False,Active,BATCH_001,employee_app,2026-09-26T12:44:46.516Z


Department,count
Support,10
DevOps,8
Finance,7
Legal,6
Operations,6
IT,5
Marketing,5
Admin,5
HR,4
Sales,3


In [0]:
department_wise_count.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.challenge_gold_department_wise_poor_performance_active_employee")

### C4 - Group Employees by Age
Business question:

How many employees belong to each age group?

In [0]:
from pyspark.sql.functions import when
employee_group = df_gold.withColumn(
    "Age_Group",
    when(col("Age").cast("int")<=30,"Under_30")
    .when(col("Age").cast("int").between(31,40),"Age_31_40")
    .when(col("Age").cast("int").between(41,50),"Age_41_50")
    .otherwise("Above_50")
)
Age_wise_employeecount = employee_group.groupBy("Age_Group").count()
display(Age_wise_employeecount)

Age_Group,count
Above_50,216
Under_30,192
Age_41_50,209
Age_31_40,206


In [0]:
Age_wise_employeecount.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.challenge_gold_age_wise_employeecount")

### C5 - Remote Workers by Department
Business question:

Which department has the highest percentage of remote workers?
Sales:
Remote = 30
Non-remote = 15
Unknown = 5

Total known = 30 + 15 = 45

Remote % = 30 / 45 × 100
         = 66.67%

In [0]:
from pyspark.sql.functions import round

remote_workers = (
    df_gold
    .filter(col("Remote_Work")=="True")
    .groupBy("Department")
    .count().withColumnRenamed("count","remote_workers")
)

known_employee_count = (
    df_gold
    .filter(col("Remote_Work")!="Unknown")
    .groupBy("Department")
    .count().withColumnRenamed("count","known_employee_count")
)

join_table = remote_workers.join(known_employee_count,on="Department",how="inner")
display(
    remote_workers
)

display(
    known_employee_count
)

final_table = join_table.withColumn(
    "percentage",
    round(
        (col("remote_workers")/col("known_employee_count"))*100,
        2
    )
)
display(final_table)

Department,remote_workers
Sales,31
Operations,29
Marketing,39
Finance,36
Admin,28
HR,33
Support,42
DevOps,27
IT,29
Legal,28


Department,known_employee_count
Sales,47
Operations,61
Marketing,64
Support,80
IT,58
Finance,69
HR,64
Admin,46
Legal,57
DevOps,65


Department,remote_workers,known_employee_count,percentage
Legal,28,57,49.12
Support,42,80,52.5
DevOps,27,65,41.54
Sales,31,47,65.96
IT,29,58,50.0
Admin,28,46,60.87
Marketing,39,64,60.94
Finance,36,69,52.17
HR,33,64,51.56
Operations,29,61,47.54


In [0]:
final_table.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.data_analysis.challenge_gold_department_wise_remoteworker"
)

### C6 - Departments Above the Company Average Salary

Business question:

Which departments have an average salary higher than the average salary of the entire company?

In [0]:
average_salary = (
    df_gold.select(avg(col("Salary").cast("double")))
).collect()[0][0]

In [0]:
average_salary

87374.07784933162

In [0]:
department_average = (
    df_gold
    .groupBy("Department")
    .agg(
        avg(col("Salary").cast("double")).alias("average_department_salary")
    )
)

In [0]:
display(
    department_average
)

Department,average_department_salary
Support,92059.80605769233
HR,84326.90137931035
Sales,85371.47953846154
Operations,83758.15511627904
Marketing,85139.08910112361
IT,97040.92040540538
Finance,82831.58103448276
Admin,93737.2047826087
Legal,84248.39650602412
DevOps,86336.95797468352


In [0]:
result = department_average.filter(
    col("average_department_salary")>average_salary
)

display(result)

Department,average_department_salary
Support,92059.80605769233
IT,97040.92040540538
Admin,93737.2047826087


In [0]:
result.write.format("delta").mode("overwrite").saveAsTable("workspace.data_analysis.challenge_gold_highest_department_salary")